# 04. Noisy Test Generation

Notebook này ghi lại kết quả của **Stage 4 — Noisy Test Generation**.

Vai trò của notebook:

- Không sinh lại dữ liệu noisy.
- Không train model.
- Chỉ đọc output đã sinh ra từ `scripts/generate_noisy_tests.py`.
- Kiểm tra số dòng, mức độ thay đổi và ví dụ clean/noisy.
- Ghi nhận các biến thể noise dùng cho robustness evaluation.

Nguyên tắc thiết kế:

```text
- Chỉ làm nhiễu test set.
- Train và validation giữ nguyên.
- Label sentiment/topic được giữ nguyên từ clean test.
- Noisy test dùng để đánh giá độ bền, không dùng để train baseline/PhoBERT.
```

## 1. Setup

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, Image

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd

print("Project root:", ROOT)

NOISY_DIR = ROOT / "data" / "noisy"
TABLES_DIR = ROOT / "outputs" / "tables"
FIGURES_DIR = ROOT / "outputs" / "figures"
REPORTS_DIR = ROOT / "outputs" / "reports"
PROCESSED_DIR = ROOT / "data" / "processed"

## 2. Kiểm tra output bắt buộc

Các file này phải tồn tại sau khi chạy:

```powershell
python scripts/generate_noisy_tests.py
```

In [ ]:
required_files = {
    "clean_test": NOISY_DIR / "test_clean.csv",
    "noisy_all": NOISY_DIR / "test_noisy_all.csv",
    "eval_all": NOISY_DIR / "test_eval_all.csv",
    "summary": TABLES_DIR / "noisy_generation_summary.csv",
    "examples": TABLES_DIR / "noisy_text_examples.csv",
    "changed_percent_figure": FIGURES_DIR / "noisy_changed_percent_by_variant.png",
    "char_change_figure": FIGURES_DIR / "noisy_char_change_ratio_by_variant.png",
    "report": REPORTS_DIR / "noisy_generation_report.md",
    "config_snapshot": REPORTS_DIR / "noise_config_snapshot.json",
}

check_df = pd.DataFrame(
    [{"name": name, "path": str(path), "exists": path.exists()} for name, path in required_files.items()]
)

display(check_df)

missing = check_df.loc[~check_df["exists"], "name"].tolist()
if missing:
    raise FileNotFoundError(f"Missing required Stage 4 files: {missing}")

print("All required Stage 4 files exist.")

## 3. Kiểm tra số dòng theo variant

Mỗi variant phải có đúng số dòng bằng test set gốc.

Nếu test set có 3166 mẫu và có 7 variants gồm clean + 6 noisy variants, `test_eval_all.csv` phải có:

```text
3166 × 7 = 22162 dòng
```

In [ ]:
eval_all = pd.read_csv(NOISY_DIR / "test_eval_all.csv")
clean_test = pd.read_csv(NOISY_DIR / "test_clean.csv")
noisy_all = pd.read_csv(NOISY_DIR / "test_noisy_all.csv")

variant_counts = eval_all.groupby("variant").size().reset_index(name="num_rows")
display(variant_counts)

expected_rows_per_variant = len(clean_test)
invalid_counts = variant_counts[variant_counts["num_rows"] != expected_rows_per_variant]

print("Rows per variant expected:", expected_rows_per_variant)
print("Total eval rows:", len(eval_all))

if len(invalid_counts) > 0:
    raise ValueError("Some variants do not have the same row count as clean test.")

print("Variant row counts are valid.")

## 4. Noisy generation summary

Bảng này cho biết mỗi biến thể noise thay đổi bao nhiêu phần trăm mẫu và mức thay đổi ký tự trung bình.

In [ ]:
summary_df = pd.read_csv(TABLES_DIR / "noisy_generation_summary.csv")
display(summary_df)

## 5. Biểu đồ mức thay đổi theo variant

In [ ]:
display(Image(filename=str(FIGURES_DIR / "noisy_changed_percent_by_variant.png")))
display(Image(filename=str(FIGURES_DIR / "noisy_char_change_ratio_by_variant.png")))

## 6. Đánh giá từng nhóm noise

Kỳ vọng thiết kế:

```text
clean:
- changed_percent = 0
- avg_char_change_ratio = 0

no_accent:
- changed gần 100%
- avg_char_change_ratio cao
- đây là biến thể mạnh nhưng phổ biến với tiếng Việt phi chuẩn

typo_light / typo_medium:
- changed_percent tăng theo mức độ
- avg_char_change_ratio thấp
- mô phỏng lỗi gõ nhẹ, không phá nghĩa câu quá nhiều

teencode_light:
- changed vừa phải
- thay các cụm như sinh viên → sv, giảng viên → gv, không → ko

mixed_light:
- kết hợp typo + teencode
- không bỏ dấu toàn bộ
- phải nhẹ hơn no_accent

mixed_no_accent:
- kết hợp bỏ dấu + typo + teencode
- là biến thể mạnh hơn mixed_light
```

In [ ]:
def get_metric(variant: str, col: str):
    rows = summary_df[summary_df["variant"] == variant]
    if len(rows) == 0:
        return None
    return float(rows.iloc[0][col])

for variant in summary_df["variant"].tolist():
    changed = get_metric(variant, "changed_percent")
    ratio = get_metric(variant, "avg_char_change_ratio")
    display(Markdown(f"- `{variant}`: changed = **{changed:.4f}%**, avg_char_change_ratio = **{ratio:.6f}**"))

# Một số kiểm tra logic nhẹ
if "clean" in summary_df["variant"].values:
    clean_changed = get_metric("clean", "changed_percent")
    if clean_changed != 0:
        display(Markdown("⚠️ `clean` should have 0% changed text."))

if "mixed_light" in summary_df["variant"].values and "no_accent" in summary_df["variant"].values:
    mixed_light_ratio = get_metric("mixed_light", "avg_char_change_ratio")
    no_accent_ratio = get_metric("no_accent", "avg_char_change_ratio")
    if mixed_light_ratio is not None and no_accent_ratio is not None:
        if mixed_light_ratio >= no_accent_ratio * 0.8:
            display(Markdown("⚠️ `mixed_light` is close to `no_accent`. Check whether `remove_diacritics` is accidentally enabled."))
        else:
            display(Markdown("`mixed_light` is clearly lighter than `no_accent`, which is expected."))

## 7. Ví dụ noisy text

Bảng này dùng để kiểm tra thủ công xem noise có làm hỏng nghĩa hoặc làm sai nhãn không.

In [ ]:
examples_df = pd.read_csv(TABLES_DIR / "noisy_text_examples.csv")
display(examples_df.head(50))

## 8. Xem ví dụ theo từng variant

In [ ]:
for variant in examples_df["variant"].drop_duplicates().tolist():
    display(Markdown(f"### Variant: `{variant}`"))
    cols = [
        "original_text",
        "text",
        "char_change_ratio",
        "sentiment_name",
        "topic_name",
    ]
    display(examples_df[examples_df["variant"] == variant][cols].head(10))

## 9. Kiểm tra label preservation

Noisy generation không được đổi nhãn. Các cột label phải được giữ nguyên từ test set gốc.

Ở đây kiểm tra mỗi `original_id` có label nhất quán giữa clean và các noisy variants.

In [ ]:
label_cols = ["sentiment_label", "sentiment_name", "topic_label", "topic_name"]

label_check = (
    eval_all.groupby("original_id")[label_cols]
    .nunique()
    .reset_index()
)

violations = label_check[
    (label_check[label_cols] > 1).any(axis=1)
]

print("Label preservation violations:", len(violations))
display(violations.head())

if len(violations) > 0:
    raise ValueError("Some noisy samples changed labels unexpectedly.")

print("Labels are preserved across clean/noisy variants.")

## 10. Kiểm tra duplicate variant id

Mỗi dòng trong `test_eval_all.csv` nên có `id` riêng.

In [ ]:
num_rows = len(eval_all)
num_unique_ids = eval_all["id"].nunique()

print("Rows:", num_rows)
print("Unique ids:", num_unique_ids)

if num_rows != num_unique_ids:
    duplicated_ids = eval_all[eval_all["id"].duplicated(keep=False)]["id"].value_counts()
    display(duplicated_ids.head())
    raise ValueError("Duplicate ids found in test_eval_all.csv")

print("No duplicate ids found.")

## 11. Kết luận Stage 4

Ghi nhận cho báo cáo:

```text
Stage 4 tạo các phiên bản noisy test set nhằm đánh giá độ bền của mô hình khi gặp tiếng Việt phi chuẩn.
Các loại nhiễu gồm bỏ dấu, lỗi gõ, teencode và nhiễu hỗn hợp.
Chỉ test set được làm nhiễu, train/validation giữ nguyên để mô phỏng tình huống mô hình được học từ dữ liệu sạch nhưng phải xử lý input phi chuẩn.
Các nhãn sentiment và topic được giữ nguyên vì noise chỉ thay đổi bề mặt văn bản, không chủ đích thay đổi ý nghĩa phản hồi.
```

Các biến thể dùng cho evaluation:

```text
clean
no_accent
typo_light
typo_medium
teencode_light
mixed_light
mixed_no_accent
```

Ý nghĩa cho giai đoạn sau:

```text
- Baseline và PhoBERT sẽ được evaluate trên cùng clean/noisy test variants.
- Robustness được đo bằng mức giảm Macro-F1 so với clean test.
- Char-level baseline có thể bền hơn trước typo/teencode.
- PhoBERT cần được kiểm tra kỹ trên no_accent và mixed_no_accent.
```

In [ ]:
display(Markdown("### Final Stage 4 summary"))
display(summary_df)

display(Markdown("### Files for next stage"))
for path in [
    NOISY_DIR / "test_eval_all.csv",
    NOISY_DIR / "test_noisy_all.csv",
    TABLES_DIR / "noisy_generation_summary.csv",
    TABLES_DIR / "noisy_text_examples.csv",
]:
    display(Markdown(f"- `{path.relative_to(ROOT)}`"))

## 12. Stage 4 status

Stage 4 hoàn thành khi:

```text
- test_eval_all.csv có đủ clean + noisy variants.
- Mỗi variant có đúng số dòng bằng test set gốc.
- clean có changed_percent = 0.
- no_accent và mixed_no_accent là nhóm noise mạnh.
- mixed_light nhẹ hơn no_accent.
- Label sentiment/topic được giữ nguyên.
- Có bảng summary, bảng examples, biểu đồ và report.
```

Giai đoạn tiếp theo đề xuất:

```text
Stage 5 — Baseline Robustness Evaluation
```

Lý do:

```text
Đánh giá lại các baseline đã train trên clean data bằng noisy test set.
Sau đó mới đưa PhoBERT lên Kaggle để có cùng chuẩn so sánh clean/noisy.
```